# Sourav's Wedding Gallery - Face Processor

This notebook will download your wedding photos, detect faces, group similar people together, and generate the files needed for the **"Find by Face"** feature.

### Step 1: Install Dependencies

In [ ]:
!pip install face_recognition requests numpy Pillow scikit-learn

### Step 2: Upload `manifest.json`
Please upload the `manifest.json` file from your project's `public/` folder.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

if 'manifest.json' not in uploaded:
    print("❌ Please upload 'manifest.json'!")
else:
    print("✅ manifest.json uploaded successfully.")

### Step 3: Process Images
This will download the images from Cloudinary and analyze them. It might take a few minutes depending on the number of photos.

In [ ]:
import json
import requests
import numpy as np
from PIL import Image
import face_recognition
from sklearn.cluster import DBSCAN
import shutil

# Configuration
MANIFEST_PATH = "manifest.json"
OUTPUT_FACES_PATH = "faces.json"
THUMBNAILS_DIR = "face-thumbnails"
IMAGES_DIR = "downloaded_images"

if not os.path.exists(THUMBNAILS_DIR):
    os.makedirs(THUMBNAILS_DIR)
if not os.path.exists(IMAGES_DIR):
    os.makedirs(IMAGES_DIR)

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

known_encodings = []
image_data = []

print(f"Processing {len(manifest)} images...")

for item in manifest:
    public_id = item['public_id']
    url = item['url']
    filename = public_id.replace('/', '_') + ".jpg"
    filepath = os.path.join(IMAGES_DIR, filename)

    if not os.path.exists(filepath):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(r.content)
        except:
            continue

    if os.path.exists(filepath):
        image = face_recognition.load_image_file(filepath)
        face_locations = face_recognition.face_locations(image)
        face_encodings = face_recognition.face_encodings(image, face_locations)

        for loc, enc in zip(face_locations, face_encodings):
            known_encodings.append(enc)
            image_data.append({"public_id": public_id, "location": loc, "filepath": filepath})

if known_encodings:
    print(f"Found {len(known_encodings)} faces. Clustering...")
    clt = DBSCAN(metric="euclidean", n_jobs=-1)
    clt.fit(known_encodings)
    label_ids = np.unique(clt.labels_)

    faces_result = []
    for label_id in label_ids:
        if label_id == -1: continue
        indexes = np.where(clt.labels_ == label_id)[0]
        rep_face = image_data[indexes[0]]
        
        img = Image.open(rep_face['filepath'])
        t, r, b, l = rep_face['location']
        pad_h, pad_w = (b-t)//2, (r-l)//2
        face_img = img.crop((max(0, l-pad_w), max(0, t-pad_h), min(img.width, r+pad_w), min(img.height, b+pad_h)))
        face_img.thumbnail((200, 200))
        
        thumb_name = f"face_{label_id}.jpg"
        face_img.save(os.path.join(THUMBNAILS_DIR, thumb_name))
        
        photos = list(set([image_data[idx]['public_id'] for idx in indexes]))
        faces_result.append({"id": int(label_id), "name": f"Person {label_id+1}", "thumbnail": f"/face-thumbnails/{thumb_name}", "photos": photos})

    with open(OUTPUT_FACES_PATH, 'w') as f:
        json.dump(faces_result, f, indent=2)
    print("✅ Processing complete!")
else:
    print("❌ No faces found.")

### Step 4: Download Results
This will create a zip file containing `faces.json` and the `face-thumbnails` folder. Download it, unzip it, and place the contents in your project's `public/` directory.

In [ ]:
import shutil
from google.colab import files

if os.path.exists(OUTPUT_FACES_PATH):
    # Create a temporary folder to zip
    os.makedirs("results", exist_ok=True)
    shutil.copy(OUTPUT_FACES_PATH, "results/faces.json")
    if os.path.exists("results/face-thumbnails"):
        shutil.rmtree("results/face-thumbnails")
    shutil.copytree(THUMBNAILS_DIR, "results/face-thumbnails")
    
    shutil.make_archive("gallery_faces", 'zip', "results")
    files.download("gallery_faces.zip")
    print("✅ Download started! Extract this zip into your project's 'public' folder.")
else:
    print("❌ No results found to download.")